# Análise da Geração de Energia Solar

## 1. Inspeção inicial

In [7]:
import pandas as pd

In [8]:
# Carregamento dos dados
df_plant_1_generation = pd.read_csv('../datasets/Plant_1_Generation_Data.csv')
df_plant_1_weather = pd.read_csv('../datasets/Plant_1_Weather_Sensor_Data.csv')
df_plant_2_generation = pd.read_csv('../datasets/Plant_2_Generation_Data.csv')
df_plant_2_weather = pd.read_csv('../datasets/Plant_2_Weather_Sensor_Data.csv')

### 1. Dados de geração da usina 1

In [10]:
df_plant_1_generation.shape

(68778, 7)

In [11]:
df_plant_1_generation.columns

Index(['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER',
       'DAILY_YIELD', 'TOTAL_YIELD'],
      dtype='str')

In [12]:
df_plant_1_generation.dtypes

DATE_TIME          str
PLANT_ID         int64
SOURCE_KEY         str
DC_POWER       float64
AC_POWER       float64
DAILY_YIELD    float64
TOTAL_YIELD    float64
dtype: object

In [13]:
df_plant_1_generation

,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.000,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.000,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.000,6987759.0
3,15-05-2020 00:00,4135001,7JYdWkrLSPkdwr4,0.0,0.0,0.000,7602960.0
4,15-05-2020 00:00,4135001,McdE0feGgRqW7Ca,0.0,0.0,0.000,7158964.0
...,...,...,...,...,...,...,...
68773,17-06-2020 23:45,4135001,uHbuxQJl8lW7ozc,0.0,0.0,5967.000,7287002.0
68774,17-06-2020 23:45,4135001,wCURE6d3bPkepu2,0.0,0.0,5147.625,7028601.0
68775,17-06-2020 23:45,4135001,z9Y9gH1T5YWrNuG,0.0,0.0,5819.000,7251204.0
68776,17-06-2020 23:45,4135001,zBIq5rxdHJRwDNY,0.0,0.0,5817.000,6583369.0


In [14]:
df_plant_1_generation.index

RangeIndex(start=0, stop=68778, step=1)

In [19]:
df_plant_1_generation["PLANT_ID"].nunique()

1

In [16]:
df_plant_1_generation["SOURCE_KEY"].nunique()

22

In [17]:
df_plant_1_generation["DATE_TIME"].nunique()

3158

In [22]:
df_plant_1_generation.duplicated().sum()

np.int64(0)

In [23]:
df_plant_1_generation.duplicated(
    subset=["PLANT_ID", "SOURCE_KEY", "DATE_TIME"]
).sum()

np.int64(0)

In [29]:
df_plant_1_generation["DATE_TIME"].dtype

<StringDtype(storage='python', na_value=nan)>

#### Observações iniciais

- O conjunto possui 68778 linhas e 7 colunas.

- Cada linha parece representar o registro feito por um microinversor específico da primeira usina em um intervalo de tempo de 15 minutos.

- O arquivo contém um único valor distinto em PLANT_ID, evidenciando que todos os registros realmente são da mesma usina.

- Foram identificados 22 inversores distintos em SOURCE_KEY.

- A coluna DATE_TIME foi interpretada como string.

- A combinação das colunas PLANT_ID, SOURCE_KEY e DATE_TIME não apresenta duplicatas, portanto é uma candidata à identificação única de cada medição.

- DC_POWER parece representar a potência em corrente contínua produzida pelos painéis.

- AC_POWER parece representar a potência em corrente alternada após a conversão pelo inversor.

- A coluna DAILY_YIELD parece representar a energia acumulada ao longo do dia até o momento da medição, já TOTAL_YIELD parece representar a energia total acumulada pelo inversor.

#### Investigação da frequência temporal dos registros

Considerando medições em intervalos de 15 minutos, seriam esperados 3264 horários distintos durante esse período. Porém, foram encontrados apenas 3158 horários, indicando a ausência de 106 horários de medição. Além disso, se todos os 22 inversores tivessem registros em todos esses horários, o esperado seria 69476 registros, mas o conjunto contém apenas 68778. Sendo assim, faz-se necessário investigar esses pontos com mais profundidade.

In [ ]:
# Criação de uma cópia do dataset com a coluna date time convertida para o formato correto

df_plant_1_generation_analysis = (
    df_plant_1_generation.copy()
)

df_plant_1_generation_analysis['DATE_TIME'] = (
    pd.to_datetime(
        df_plant_1_generation_analysis['DATE_TIME'], format="%d-%m-%Y %H:%M"
    )
)

In [54]:
df_plant_1_generation_analysis['DATE_TIME'].min()

Timestamp('2020-05-15 00:00:00')

In [55]:
df_plant_1_generation_analysis['DATE_TIME'].max()

Timestamp('2020-06-17 23:45:00')

In [40]:
df_plant_1_generation["SOURCE_KEY"].value_counts()

SOURCE_KEY
bvBOhCH3iADSZry    3155
1BY6WEcLGh8j5v7    3154
7JYdWkrLSPkdwr4    3133
VHMLBKoKgIrUVDU    3133
ZnxXDlPa8U1GXgE    3130
ih0vzX44oOqAx2f    3130
wCURE6d3bPkepu2    3126
z9Y9gH1T5YWrNuG    3126
iCRJl6heRkivqQ3    3125
pkci93gMrogZuBj    3125
uHbuxQJl8lW7ozc    3125
McdE0feGgRqW7Ca    3124
rGa61gmuvPhdLxV    3124
sjndEbLyjtCKgGv    3124
zVJPv84UY57bAof    3124
ZoEaEvLYb1n2sOq    3123
1IF53ai7Xc0U56Y    3119
adLQvlD726eNBSB    3119
zBIq5rxdHJRwDNY    3119
3PZuoBAID5Wc2HD    3118
WRmjgnKYAwPKWDb    3118
YxYtjZvoooNbGkE    3104
Name: count, dtype: int64

In [57]:
unique_date_times = df_plant_1_generation_analysis['DATE_TIME'].drop_duplicates().sort_values()

In [58]:
time_intervals = unique_date_times.diff()

In [59]:
time_intervals.value_counts()

DATE_TIME
0 days 00:15:00    3148
0 days 00:30:00       2
0 days 03:00:00       1
0 days 01:00:00       1
0 days 04:15:00       1
0 days 09:00:00       1
0 days 01:45:00       1
0 days 08:00:00       1
0 days 00:45:00       1
Name: count, dtype: int64

In [50]:
time_gap_analysis = pd.DataFrame({
    'previous_date_time': unique_date_times.shift(1),
    'current_date_time': unique_date_times,
    'interval': time_intervals
})

time_gap_analysis[
    time_gap_analysis['interval'] > pd.Timedelta(minutes=15)
]

,previous_date_time,current_date_time,interval
1954,2020-05-15 23:00:00,2020-05-16 02:00:00,0 days 03:00:00
9146,2020-05-19 11:30:00,2020-05-19 12:30:00,0 days 01:00:00
11290,2020-05-20 13:15:00,2020-05-20 17:30:00,0 days 04:15:00
11774,2020-05-20 22:45:00,2020-05-21 07:45:00,0 days 09:00:00
15632,2020-05-23 05:00:00,2020-05-23 06:45:00,0 days 01:45:00
16952,2020-05-23 21:30:00,2020-05-23 22:00:00,0 days 00:30:00
19728,2020-05-25 05:30:00,2020-05-25 06:00:00,0 days 00:30:00
27404,2020-05-28 22:15:00,2020-05-29 06:15:00,0 days 08:00:00
67260,2020-06-17 06:00:00,2020-06-17 06:45:00,0 days 00:45:00


In [56]:
inverter_periods = (
    df_plant_1_generation_analysis
    .groupby('SOURCE_KEY')['DATE_TIME']
    .agg(['min', 'max', 'count'])
)

inverter_periods

,min,max,count
SOURCE_KEY,,,
1BY6WEcLGh8j5v7,2020-05-15 00:00:00,2020-06-17 23:45:00,3154
1IF53ai7Xc0U56Y,2020-05-15 00:00:00,2020-06-17 23:45:00,3119
3PZuoBAID5Wc2HD,2020-05-15 00:00:00,2020-06-17 23:45:00,3118
7JYdWkrLSPkdwr4,2020-05-15 00:00:00,2020-06-17 23:45:00,3133
McdE0feGgRqW7Ca,2020-05-15 00:00:00,2020-06-17 23:45:00,3124
VHMLBKoKgIrUVDU,2020-05-15 00:00:00,2020-06-17 23:45:00,3133
WRmjgnKYAwPKWDb,2020-05-15 00:00:00,2020-06-17 23:45:00,3118
YxYtjZvoooNbGkE,2020-05-15 01:00:00,2020-06-17 23:45:00,3104
ZnxXDlPa8U1GXgE,2020-05-15 00:00:00,2020-06-17 23:45:00,3130


#### Observações da investigação de frequência temporal

- O conjunto de dados abrange o período entre 15/05/2020 às 00:00 e 17/06/2020 às 23:45, totalizando 34 dias registrados.

- A maior parte dos horários consecutivos apresenta o intervalo esperado de 15 minutos. Entretanto, foram identificadas nove lacunas temporais maiores, variando entre 30 minutos e 9 horas. Desse modo, fica evidente porque existem exatamente os 106 horários ausentes no período analisado.

- As maiores interrupções ocorreram entre 20/05/2020 às 22:45 e 21/05/2020 às 07:45, com duração de 9 horas, e entre 28/05/2020 às 22:15 e 29/05/2020 às 06:15, com duração de 8 horas.

- A quantidade de registros por inversor também varia entre os 22 inversores. O inversor com mais medições possui 3.155 registros, enquanto o inversor com menos medições possui 3.104 registros.

- Existem 698 combinações ausentes entre inversor e horário.

- Com exceção do inversor 'YxYtjZvoooNbGkE', que apresenta o primeiro registro em 15/05/2020 às 01:00, todos os inversores apresentam registros desde 15/05/2020 às 00:00. Todos possuem como último registro o horário de 17/06/2020 às 23:45. Sendo assim, as diferenças na quantidade de medições não parece ser originada por períodos de funcionamento distintos entre inversores, mas sim estar relacionados principalmente a ausências distribuidas ao longo do período observado.

#### Verificação de qualidade dos valores

In [61]:
missing_values = df_plant_1_generation.isna().sum()

missing_values[
    missing_values > 0
]

Series([], dtype: int64)

In [63]:
numeric_columns = [
    'DC_POWER',
    'AC_POWER',
    'DAILY_YIELD',
    'TOTAL_YIELD'
]

In [65]:
(df_plant_1_generation[numeric_columns] < 0).sum()

DC_POWER       0
AC_POWER       0
DAILY_YIELD    0
TOTAL_YIELD    0
dtype: int64

In [66]:
(df_plant_1_generation[numeric_columns] == 0).sum()

DC_POWER       31951
AC_POWER       31951
DAILY_YIELD    18696
TOTAL_YIELD        0
dtype: int64

In [ ]:
# Verificando a porcentagem do conjunto que é igual a zero

(df_plant_1_generation[numeric_columns] == 0).sum() / df_plant_1_generation.shape[0] * 100

DC_POWER       46.455262
AC_POWER       46.455262
DAILY_YIELD    27.183111
TOTAL_YIELD     0.000000
dtype: float64

In [64]:
df_plant_1_generation[numeric_columns].describe()

,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
count,68778.000000,68778.000000,68778.000000,6.877800e+04
mean,3147.426211,307.802752,3295.968737,6.978712e+06
std,4036.457169,394.396439,3145.178309,4.162720e+05
min,0.000000,0.000000,0.000000,6.183645e+06
25%,0.000000,0.000000,0.000000,6.512003e+06
50%,429.000000,41.493750,2658.714286,7.146685e+06
75%,6366.964286,623.618750,6274.000000,7.268706e+06
max,14471.125000,1410.950000,9163.000000,7.846821e+06


#### Resultado inicial da qualidade dos valores

- O conjunto não possui valores ausentes explícitos nem valores negativos.

- As colunas DC_POWER e AC_POWER possuem 31.951 registros iguais a zero, correspondentes a aproximadamente 46.46% do conjunto. Provavelmente, parte desses valores está associada aos períodos sem geração solar.

- A média de DC_POWER é aproximadamente 3147.43, enquanto sua mediana é 429.00. Em AC_POWER, a média é aproximadamente 307.80 e a mediana é 41.49. A diferença entre média e mediana, juntamente com a grande quantidade de zeros, indica que existe uma grande concentração de valores baixos e presença de medições consideravelmente mais elevadas que puxam a média para cima.

- Os valores de DC_POWER e AC_POWER apresentam escalas bem distintas, o que pode indicar perda na conversão, mas ainda é necessário analisar mais a fundo.

- A coluna DAILY_YIELD possui 18.696 registros iguais a zero, correspondentes a aproximadamente 27.18% do conjunto. Provavelmente esses registros estão relacionados ao início do ciclo de geração diário.

- A colunas TOTAL_YIELD não possui valores iguais a zero. Seus valores variam entre aproximadamente 6.18 milhões e 7.85 milhões.